# People Analytics — Warehouse Explorer

Interactive notebook for poking at the DuckDB warehouse built by `uv run pa-setup`.

**Run order:** Use `Shift+Enter` to run a cell. Pick the `.venv` kernel when prompted (top-right of this notebook).

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

DB = Path('..') / 'warehouse' / 'people.duckdb'
con = duckdb.connect(str(DB), read_only=True)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 200)
print(f'Connected to {DB.resolve()}')

## What's in the warehouse?

In [ ]:
con.sql("""
    select table_schema, table_name, table_type
    from information_schema.tables
    where table_schema not in ('information_schema', 'pg_catalog')
    order by table_schema, table_name
""")

## Workforce metrics — last 14 days of data

In [ ]:
con.sql("""
    select date_day, active_headcount, total_fte, hires, terminations, net_headcount_change
    from marts.mart_workforce_metrics_daily
    where active_headcount > 0
    order by date_day desc
    limit 14
""")

## Hiring + attrition by month (chartable)

In [ ]:
monthly = con.sql("""
    select
        date_trunc('month', date_day)::date as month,
        sum(hires) as hires,
        sum(terminations) as terms,
        sum(hires) - sum(terminations) as net,
        max(active_headcount) as eom_headcount
    from marts.mart_workforce_metrics_daily
    where active_headcount > 0
    group by 1
    order by 1
""").df()
monthly

In [ ]:
ax = monthly.set_index('month')[['hires', 'terms']].plot(kind='bar', figsize=(12, 4), title='Hires vs terminations by month')
ax.set_ylabel('count');

## SCD2 in action — a worker with multiple profile versions

In [ ]:
con.sql("""
    with multi as (
        select worker_id
        from core.dim_employee_profile_scd
        group by worker_id
        having count(*) >= 4
        limit 1
    )
    select
        s.worker_id,
        s.version_number,
        s.valid_from_date,
        s.valid_to_date,
        s.department,
        s.job_title,
        s.job_level,
        s.version_event_type
    from core.dim_employee_profile_scd s
    join multi using (worker_id)
    order by s.valid_from_date
""")

## Rehires — same person, multiple worker_ids

In [ ]:
con.sql("""
    select
        p.person_external_id,
        p.first_name || ' ' || p.last_name as name,
        e.episode_number,
        e.worker_id,
        e.hire_date,
        e.termination_date,
        e.tenure_days
    from core.dim_employment_episode e
    join core.dim_person p using (person_key)
    where p.person_external_id in (
        select person_external_id
        from core.dim_employment_episode
        join core.dim_person using (person_key)
        group by person_external_id
        having count(*) > 1
    )
    order by p.person_external_id, e.episode_number
""")

## Org pyramid — depth distribution from recursive manager-chain CTE

In [ ]:
con.sql("""
    select depth, count(distinct employee_key) as employees
    from core.fact_manager_chain_daily
    where snapshot_date = (select max(snapshot_date) from core.fact_manager_chain_daily)
    group by depth
    order by depth
""")

## Span of control — biggest managers today

In [ ]:
con.sql("""
    with latest as (select max(snapshot_date) as d from core.fact_headcount_snapshot_daily)
    select
        m.first_name || ' ' || m.last_name as manager,
        m.current_job_title,
        m.current_department,
        count(*) as direct_reports
    from core.fact_headcount_snapshot_daily s
    join core.dim_employee m on s.manager_employee_key = m.employee_key
    cross join latest
    where s.snapshot_date = latest.d
    group by 1, 2, 3
    order by direct_reports desc
    limit 10
""")

## Sandbox — write whatever you want below

In [ ]:
con.sql("""
    select * from marts.mart_workforce_metrics_daily limit 5
""")